# LSTM using Pytorch

Build and train a simple Deep Learning Sequence architecture (RNN or LSTM) using PyTorch to understand text sequence mapping and recurrent state updates.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

## Text Sequence Mapping & Data Prep

In [2]:
text = "deep learning sequences map data over time"
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

seq_length = 4
inputs = []
targets = []

for i in range(len(text) - seq_length):
    seq_in = text[i : i + seq_length]
    seq_out = text[i + 1 : i + seq_length + 1]
    inputs.append([char_to_ix[ch] for ch in seq_in])
    targets.append([char_to_ix[ch] for ch in seq_out])

X = torch.tensor(inputs, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)

## LSTM Sequence Architecture

In [3]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(CharLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden_state=None):
        embeds = self.embedding(x)
        out, hidden_state = self.lstm(embeds, hidden_state)
        
        out = out.reshape(-1, out.shape[2])
        out = self.fc(out)
        
        return out, hidden_state

## Model Initialization & Hyperparameters

In [4]:
embedding_dim = 8
hidden_dim = 16
model = CharLSTM(vocab_size, embedding_dim, hidden_dim)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

## Training Loop

In [5]:
epochs = 150
for epoch in range(epochs):
    hidden = None 
    
    optimizer.zero_grad()
    
    outputs, hidden = model(X, hidden)
    
    hidden = (hidden[0].detach(), hidden[1].detach())
    
    loss = criterion(outputs, y.view(-1))
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 30 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [30/150], Loss: 1.8213
Epoch [60/150], Loss: 0.6816
Epoch [90/150], Loss: 0.3765
Epoch [120/150], Loss: 0.3106
Epoch [150/150], Loss: 0.2884


## Inference: Demonstrating Recurrent State Update

In [6]:
print("\n--- Generating Sequence Step-by-Step ---")
start_char = "d"
current_char_tensor = torch.tensor([[char_to_ix[start_char]]], dtype=torch.long)
generated_text = start_char
hidden = None

model.eval()
with torch.no_grad():
    for step in range(25):
        output, hidden = model(current_char_tensor, hidden)

        predicted_ix = int(torch.argmax(output).item())
        predicted_char = ix_to_char[predicted_ix]
        generated_text += predicted_char

        current_char_tensor = torch.tensor([[predicted_ix]], dtype=torch.long)

print(f"Generated text: '{generated_text}'")


--- Generating Sequence Step-by-Step ---
Generated text: 'data over timees map data '
